# 05b · Preparación Gold: filtro de Kalman (Propuesta B)

| | |
|---|---|
| **Fase CRISP-DM** | 3 — Preparación de los datos |
| **Objetivo** | Filtrar las series con un filtro de Kalman (Propuesta B), alternativa a BO-VMD, para el modelo Seq2Seq con atención del notebook 06b. |
| **Entradas** | `data/silver/voltajes_silver.parquet`, `data/silver/split.json`, `models/vmd_params.json` (misma muestra de celdas que BO-VMD) |
| **Salidas** | `data/gold/series_kalman.parquet`, `data/gold/escaladores_kalman.json`, `models/kalman_params.json`, figuras en `reports/gold/` |

**Modelo de espacio de estados (nivel y tendencia local)**
- Estado `x_t = [nivel, pendiente]`, con `nivel_t = nivel_{t−1} + pendiente_{t−1} + w₁` y `pendiente_t = pendiente_{t−1} + w₂`.
- Observación `y_t = nivel_t + v`, con varianzas `q_nivel`, `q_pend` (proceso) y `r` (medición).
- **Estimación:** máxima verosimilitud **solo con train**, sobre la misma muestra de 20 celdas que BO-VMD.
- **Filtrado:** el filtro es **causal por construcción** (la estimación en `t` usa solo datos ≤ `t`), así que se aplica a la serie completa sin fuga de información y no necesita recalcularse por origen, a diferencia de VMD. Los tramos `fuera_servicio` reinician el filtro; los días sin dato válido solo se predicen.

In [ ]:
# Configuración: localiza la raíz del proyecto (local o Colab) y agrega src/ al path
import sys
from pathlib import Path

try:
    import google.colab  # noqa: F401
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/MyDrive/Poc-TP1")
except ImportError:
    ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "voltcast").is_dir())
sys.path.insert(0, str(ROOT / "src"))

In [ ]:
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from voltcast import cleaning as cl
from voltcast import config, io
from voltcast import dataset as ds
from voltcast import kalman as kf

config.ensure_dirs()
FIG_DIR = config.REPORTS_DIR / "gold"
FIG_DIR.mkdir(parents=True, exist_ok=True)
sns.set_theme(style="whitegrid", context="notebook")


def savefig(fig, name):
    path = FIG_DIR / f"{name}.png"
    fig.savefig(path, dpi=130, bbox_inches="tight")
    print(f"Figura guardada: {path.relative_to(config.BASE_DIR)}")


silver = io.read_layer("silver", "voltajes_silver.parquet")
split = io.read_json("silver", "split.json")
corte = pd.Timestamp(split["fecha_corte"])
CELLS = sorted(silver["celda"].astype(str).unique())
V = ds.pivot(silver, "voltaje", CELLS)
Q = ds.pivot(silver, "calidad", CELLS)
Y = V.where(Q.isin(cl.CALIDAD_VALIDA)).to_numpy()       # observaciones: solo calidad válida
OOS = (Q == "fuera_servicio").to_numpy()
es_train = V.index < corte
print(f"{len(CELLS)} celdas × {len(V)} días | train hasta {split['train_hasta']}")

## 1. Estimación de las varianzas por máxima verosimilitud (solo train)

Los **saltos de nivel** (intervenciones) y los **arranques tras un paro** son cambios bruscos que el modelo de nivel y tendencia local interpretaría como ruido de proceso. Si se los deja, `q_nivel` se infla y `r` cae a ~0: el filtro deja pasar la señal tal cual. Para la **estimación**, el filtro se reinicia en esos días (saltos detectados con la regla R9 solo sobre train). Al **filtrar** no se usan los saltos, porque detectarlos requiere conocer los 7 días siguientes.

In [ ]:
Vtr, Qtr = V.loc[es_train], Q.loc[es_train]
saltos_tr = cl.level_shifts(Vtr, Qtr.apply(lambda s: pd.Categorical(s, categories=cl.CALIDAD).codes))
reinicio = OOS[es_train].copy()
pos = {d: i for i, d in enumerate(Vtr.index)}
for _, r in saltos_tr.iterrows():
    reinicio[pos[r["fecha"]], CELLS.index(r["celda"])] = True
paro = (Qtr == "paro").to_numpy()
reinicio[1:] |= paro[:-1] & ~paro[1:]  # primer día tras un paro

muestra = [m["celda"] for m in json.loads((config.MODELS_DIR / "vmd_params.json").read_text(encoding="utf-8"))["muestra"]]
ajustes = {}
for grupo, celdas in [("A", [c for c in muestra if c[0] == "A"]), ("B", [c for c in muestra if c[0] == "B"]), ("común", muestra)]:
    j = [CELLS.index(c) for c in celdas]
    ajustes[grupo] = kf.fit_mle(Y[es_train][:, j], reinicio[:, j])
    # Referencia: estimación sin reiniciar en saltos ni arranques
ajustes["común (sin reinicios)"] = kf.fit_mle(Y[es_train][:, [CELLS.index(c) for c in muestra]],
                                               OOS[es_train][:, [CELLS.index(c) for c in muestra]])
tabla = pd.DataFrame(ajustes).T
for c in ["q_nivel", "q_pend", "r"]:
    tabla[f"σ {c} (V)"] = np.sqrt(tabla[c].astype(float))
tabla[["σ q_nivel (V)", "σ q_pend (V)", "σ r (V)", "loglik", "convergio"]]

In [ ]:
PARAMS = {k: float(ajustes["común"][k]) for k in ("q_nivel", "q_pend", "r")}
ratio = {k: float(ajustes["A"][k] / ajustes["B"][k]) for k in ("q_nivel", "r")}
print(f"Cociente A/B: q_nivel {ratio['q_nivel']:.2f}, r {ratio['r']:.2f} → se usan los parámetros comunes")
print(f"Relación señal/ruido q_nivel / r = {PARAMS['q_nivel'] / PARAMS['r']:.2f}")
kalman_params = {**PARAMS, "modelo": "nivel y tendencia local", "estimacion": "MV conjunta sobre la muestra BO (train)",
                 "reinicios_en_estimacion": "saltos de nivel (train) y primer día tras paro",
                 "ajustes": {k: {kk: (float(vv) if isinstance(vv, (int, float)) else vv) for kk, vv in v.items()}
                             for k, v in ajustes.items()},
                 "muestra": muestra, "entrenado_hasta": split["train_hasta"]}
(config.MODELS_DIR / "kalman_params.json").write_text(json.dumps(kalman_params, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"✔ {(config.MODELS_DIR / 'kalman_params.json').relative_to(config.BASE_DIR)}")

**Lectura.**
- **Sin reiniciar** el filtro en saltos y arranques, la máxima verosimilitud atribuye casi toda la variación al nivel (σ del cambio diario ≈ 0,012 V) y el filtro apenas suaviza.
- **Con los reinicios**, la estimación conjunta da σ del ruido de medición ≈ 0,006 V y σ del cambio diario de nivel ≈ 0,006 V (relación señal/ruido `q_nivel / r ≈ 0,85`).
- **Heterogeneidad entre electrolizadores:** con 150 celdas, el EL A y el EL B dan parámetros bastante distintos (`q_nivel` del EL A ≈ 10 veces menor, `r` ≈ 3 veces mayor). Se usa el **par común** de la estimación conjunta, como en la especificación; es una simplificación que puede restar algo de precisión al filtro en cada electrolizador y queda documentada.
- La pendiente estimada es prácticamente constante (`q_pend ≈ 0`): en la práctica el modelo se comporta como uno de nivel local.

## 2. Filtrado causal de todas las celdas

In [ ]:
KF, KS = kf.kalman_filter(Y, OOS, **PARAMS, return_slope=True)  # nivel y pendiente filtrados
resid = (Y - KF)
print(f"|real − Kalman| (fechas válidas): mediana {np.nanmedian(np.abs(resid)):.5f} V | "
      f"P95 {np.nanpercentile(np.abs(resid), 95):.5f} V | P99 {np.nanpercentile(np.abs(resid), 99):.5f} V")
print(f"Celdas×día sin estimación (fuera de servicio o antes de la primera observación): {np.isnan(KF).mean():.2%}")

# Causalidad: filtrar la serie cortada en una fecha da el mismo valor en esa fecha que filtrar la serie completa
for fecha in ["2025-04-28", "2025-10-01", "2026-03-15"]:
    t = V.index.get_loc(pd.Timestamp(fecha))
    corto = kf.kalman_filter(Y[: t + 1], OOS[: t + 1], **PARAMS)[-1]
    assert np.allclose(corto, KF[t], equal_nan=True)
print("✔ filtro causal: el valor en t no cambia al agregar datos posteriores")

In [ ]:
c = muestra[0]
j = CELLS.index(c)
desde = V.index[es_train][-365]
m = V.index >= desde
fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
axes[0].plot(V.index[m], V[c][m], color="lightgrey", label="real (Silver)")
axes[0].plot(V.index[m], KF[m, j], color="tab:green", lw=1, label="Kalman")
axes[0].axvline(corte, color="k", ls="--", lw=0.8); axes[0].legend(); axes[0].set(ylabel="V", title=f"{c}: filtro de Kalman")
axes[1].plot(V.index[m], (V[c] - KF[:, j])[m], color="tab:green", lw=0.8)
axes[1].set(ylabel="real − Kalman (V)")
fig.tight_layout(); savefig(fig, "06_kalman_ejemplo")

## 3. Ancla y σ_Δ con la señal de Kalman (solo train)

In [ ]:
KF_df = pd.DataFrame(KF, index=V.index, columns=CELLS)
KFtr = KF_df.loc[es_train]
ANC = ds.anchor(KFtr)
ok_t0 = ds.origin_ok(ANC, pd.DataFrame(OOS[es_train], index=KFtr.index, columns=CELLS))
sigma_k, por_h = ds.delta_stats(Vtr, Qtr.isin(cl.CALIDAD_VALIDA), ANC, ok_t0, config.HORIZON_DAYS)
esc_vmd = io.read_json("gold", "escaladores.json")
print(f"σ_Δ con Kalman = {sigma_k:.4f} V (con VMD: {esc_vmd['sigma_delta']:.4f} V)")

## 4. Escritura en Gold

In [ ]:
out = KF_df.rename_axis("fecha").reset_index().melt(id_vars="fecha", var_name="celda", value_name="v_kalman")
out["pendiente_kalman"] = pd.DataFrame(KS, index=V.index, columns=CELLS).to_numpy().T.ravel()  # mismo orden que melt
out["electrolizador"] = out["celda"].str[0].astype("category")
out["celda"] = out["celda"].astype("category")
src = config.SILVER_DIR / "voltajes_silver.parquet"
p = io.write_layer(out, "gold", "series_kalman.parquet", source=src, extra={"split": split, "kalman": PARAMS})
print(f"✔ {p.relative_to(config.BASE_DIR)} ({len(out):,} filas)")
esc_k = {**esc_vmd, "sigma_delta": sigma_k, "filtro": "kalman", "kalman_params": PARAMS}
esc_k.pop("vmd_params", None); esc_k.pop("vmd_n_drop", None)
p = io.write_json(esc_k, "gold", "escaladores_kalman.json")
print(f"✔ {p.relative_to(config.BASE_DIR)}")

## Conclusión

- Se generó la señal de Kalman para las 300 celdas (`data/gold/series_kalman.parquet`, nivel y pendiente), con parámetros estimados solo con train (`models/kalman_params.json`).
- El filtro es **causal**: se verificó que el valor en `t` no cambia al agregar datos posteriores. Por eso sirve para validación y despliegue sin recalcular por origen, una ventaja práctica frente a la VMD causal.
- La señal filtrada difiere de la real en ~0,0005 V de mediana. σ_Δ con Kalman (0,1306 V) es prácticamente igual que con VMD (0,1292 V).

**Siguiente:** notebook 06b — Propuesta B (Seq2Seq BiLSTM con atención + ensamble con Kalman).